# 01 - MLP NumPy

Etapa 1: MLP puro em NumPy, CrossEntropy estavel, SGD com momentum e gradient check.

In [ ]:
import sys
from pathlib import Path

for src_path in [Path.cwd() / 'src', Path.cwd().parent / 'src', Path('/content/ap2-ia/src')]:
    if src_path.exists():
        sys.path.insert(0, str(src_path))
        break

import random
import numpy as np
from medmnist import PathMNIST
from numpy_nn.model import NumpyMLP
from numpy_nn.optimizers import SGDMomentum
from numpy_nn.gradient_check import gradient_check

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

In [ ]:
train_ds = PathMNIST(split='train', size=28, download=True)
x = train_ds.imgs.astype('float32') / 255.0
x = x.mean(axis=-1)  # RGB 28x28x3 -> grayscale 28x28, conforme entrada 784 da MLP
x = x.reshape(len(x), -1)
y = train_ds.labels.reshape(-1).astype(int)

model = NumpyMLP([784, 128, 64, 9], seed=42)
diff = gradient_check(NumpyMLP([784, 8, 9], seed=42), x[:4], y[:4])
diff

In [ ]:
optimizer = SGDMomentum(model.params, lr=1e-2, beta=0.9)
batch_size = 128
history = []

for epoch in range(20):
    indices = np.random.permutation(len(x))
    losses = []
    for start in range(0, len(x), batch_size):
        batch = indices[start:start + batch_size]
        loss, grads = model.loss_and_grads(x[batch], y[batch])
        optimizer.step(model.params, grads)
        losses.append(loss)
    pred = model.predict(x[:5000])
    acc = (pred == y[:5000]).mean()
    history.append({'epoch': epoch + 1, 'loss': float(np.mean(losses)), 'acc_sample': float(acc)})
    print(history[-1])